RQs:
- What is the distribution of contract type between occupations/sectors?
- What is the distribution of salary between occupations/sectors?
    - Avg annual increase by sector?
    - Stretch: hourly pay / annual pay / annualised pay?
- What proportion of ads in each occupation/sector mention learning and development/career progression/CPD?
- What proportion of ads in each occupation/sector mention flexible hours/flexible shifts?

Stretch: unsupervised approach: topic modelling of JQ sentences in early years sector vs a comparison sector

Limitations:
- The sample only covers the years 2021-2023 inclusive

In [ ]:
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import re
from typing import List

from dap_job_quality import PROJECT_DIR
from dap_job_quality.getters.keywords import get_keywords
from dap_job_quality.utils import analysis_utils

In [ ]:
def plot_proportion(df, dim='FLEX_HOURS', save_path=None):

    filtered_df = df[df[dim] == 1.0]

    count_df = filtered_df.groupby('sector')['id'].count()

    total_count = df.groupby('sector')['id'].count()

    proportion_df = (count_df / total_count) * 100
    proportion_df = proportion_df.sort_values(ascending=False)

    _, ax = plt.subplots(figsize=(12, 8))

    bars = ax.barh(proportion_df.index, proportion_df.values, color='skyblue', edgecolor='black')

    for bar in bars:
        width = bar.get_width()
        ax.text(width + 0.5, bar.get_y() + bar.get_height()/2, f'{width:.0f}%', 
                va='center', ha='left', color='black', fontsize=10)
    
    ax.set_xlabel('Proportion (%)')
    ax.set_ylabel('Sector')
    ax.set_title(f'Proportion of job ads mentioning {dim} by Sector')
    
    if save_path:
        plt.savefig(save_path, bbox_inches='tight')

    plt.tight_layout()
    plt.show()

In [ ]:
# Produced by pipeline/eyp/stratified_sample.py
afs_raw_sample = pd.read_parquet('s3://open-jobs-lake/job_quality/early_years/evaluation_sample/job_ads_by_sector_region_sample_size_19896.parquet')
# Produced by pipeline/find_job_quality.py
processed_ads = pd.read_parquet('s3://open-jobs-lake/job_quality/outputs/afs/job_ads_prod_True_n_19896.parquet')
# Produced by pipeline/eyp/eyp_auxiliary_data.py
salary_info = pd.read_parquet('s3://open-jobs-lake/job_quality/early_years/evaluation_sample/job_ads_by_sector_region_sample_size_19896_metadata_production_True.parquet')
lookup = get_keywords()

In [ ]:
category_replacements = {
    'eyp': 'Early Years Practitioner',
    'non education': 'Waiter, Retail Assistant',
    'school teacher': 'Primary/Secondary School Teacher',
    'special needs teacher': 'Special Needs Teacher',
    'supply teacher': 'Supply Teacher',
    'teaching assistant': 'Teaching Assistant',
}

# Replace the categories in the column
afs_raw_sample['sector_group'] = afs_raw_sample['sector_group'].replace(category_replacements)

afs_raw_sample['sector_group'].value_counts()

In [ ]:
print(f"The complete sample includes {len(afs_raw_sample)} job adverts")

In [ ]:
len(processed_ads['id'].unique())

In [ ]:
print(f"The sectors to which EYP roles are compared are: {afs_raw_sample['sector'].unique()}")

Data for [this visualisation](https://app.flourish.studio/visualisation/19037076/edit):

In [ ]:
sample_sector_by_itl = afs_raw_sample.groupby(['sector', 'itl_1_name']).agg('size').unstack(fill_value=0)
sample_sector_by_itl.to_csv('outputs/sector_itl1.csv')
sample_sector_by_itl

Data for [this visualisation](https://app.flourish.studio/visualisation/19037320/edit):

In [ ]:
distribution_itl1 = afs_raw_sample.groupby(['itl_1_name']).agg('size')
distribution_itl1.to_csv('itl1.csv')
distribution_itl1

In [ ]:
distribution_sector = afs_raw_sample.groupby(['sector']).agg('size')
distribution_sector.to_csv('outputs/sector.csv')
distribution_sector

In [ ]:
processed_ads = processed_ads[['id', 'sentences_split', 'target_phrase']].drop_duplicates()

In [ ]:
processed_ads = pd.merge(processed_ads, lookup[['target_phrase', 'subcategory','dimension']], on='target_phrase', how='left')

In [ ]:
dimensions_wide = analysis_utils.create_wide_table(processed_ads)

In [ ]:
afs_sample = pd.merge(afs_raw_sample, dimensions_wide, on='id', how='left')

In [ ]:
columns_to_replace = dimensions_wide.columns[1:] # the first column is the id

In [ ]:
# These columns have NaN where there are *no* mentions of JQ dimensions in these job adverts
afs_sample[columns_to_replace] = afs_sample[columns_to_replace].fillna(0)

In [ ]:
plot_proportion(afs_sample, 'FLEX_HOURS', save_path='outputs/prop_flex_hours_by_sector.png')

In [ ]:
plot_proportion(afs_sample, 'L&D', save_path='outputs/prop_l&d_by_sector.png')

# Clean and analyse salary data

In [ ]:
# Check the distribution of different salary rates
salary_info['raw_salary_unit'].value_counts(dropna=False)

In [ ]:
missing_salary_df = salary_info[salary_info['raw_salary_unit'].isna()]

In [ ]:
analysis_utils.extract_salary_info("From £9.50 per hour")

In [ ]:
salary_descs = missing_salary_df['description'].tolist()
salaries = [analysis_utils.extract_salary_info(desc) for desc in salary_descs]
salaries

In [ ]:
# An example of an ad that includes a £3000 refer-a-friend bonus
missing_salary_df.iloc[8]['description']

In [ ]:
data = []
for sublist in salaries:
    if sublist:
        data.extend(sublist)
    else:
        data.append({'min_salary': None, 'max_salary': None, 'rate': None})

In [ ]:
extracted_salaries_df = pd.DataFrame(data)
extracted_salaries_df['salary'] = extracted_salaries_df['min_salary'].fillna(0)
extracted_salaries_df.head()

In [ ]:
missing_salary_df['raw_salary_unit'] = extracted_salaries_df['rate']
missing_salary_df['raw_salary_float'] = extracted_salaries_df['salary']
missing_salary_df['raw_min_salary_float'] = extracted_salaries_df['min_salary']
missing_salary_df['raw_max_salary_float'] = extracted_salaries_df['max_salary']

In [ ]:
enhanced_salary_data = pd.concat([missing_salary_df, salary_info[~salary_info['raw_salary_unit'].isna()]])
enhanced_salary_data['raw_salary_unit'] = enhanced_salary_data['raw_salary_unit'].str.lower()
enhanced_salary_data['raw_salary_unit'] = enhanced_salary_data['raw_salary_unit'].replace('annum', 'year')
enhanced_salary_data['raw_salary_unit'] = enhanced_salary_data['raw_salary_unit'].replace([np.nan, None, 'unknown'], np.nan)

In [ ]:
enhanced_salary_data['raw_salary_unit'].value_counts(dropna=False)

In [ ]:
# Check how many ads have the salary expressed per annum
enhanced_salary_data['is_annualised'] = enhanced_salary_data['raw_salary_unit'] == 'year'
enhanced_salary_data['is_annualised'].value_counts(dropna=False)

In [ ]:
afs_sample_enhanced = pd.merge(afs_raw_sample, enhanced_salary_data[['id', 'raw_salary_unit', 'raw_salary_float', 'raw_min_salary_float', 'raw_max_salary_float','is_annualised']], on='id', how='left')

In [ ]:
# Check that the merge was 1-1, and did not introduce duplicates
len(afs_sample_enhanced) - len(afs_raw_sample)

In [ ]:
# Filter the data to only records that have some kind of salary rate unit
afs_sample_enhanced_w_salaries = afs_sample_enhanced[afs_sample_enhanced['raw_salary_unit'].notna()]

In [ ]:
afs_sample_enhanced_w_salaries.columns

In [ ]:
# proportion of ads offering hourly/daily/yearly salary
grouped = afs_sample_enhanced_w_salaries.groupby(['sector_group', 'raw_salary_unit']).size().unstack(fill_value=0)
grouped

In [ ]:
grouped.to_csv('outputs/salary_unit_by_sector.csv')

In [ ]:
afs_sample_enhanced_w_salaries['raw_salary_float'] = afs_sample_enhanced_w_salaries['raw_salary_float'].fillna(afs_sample_enhanced_w_salaries['raw_min_salary_float'])

afs_sample_enhanced_w_salaries['hourly_wage'] = afs_sample_enhanced_w_salaries.apply(analysis_utils.calculate_hourly_wage, axis=1)

In the next chunks we check the distribution of hourly wage and exclude outlying data.

In [ ]:
afs_sample_enhanced_w_salaries['hourly_wage'].describe()

In [ ]:
afs_sample_enhanced_w_salaries[afs_sample_enhanced_w_salaries['hourly_wage']<10]['hourly_wage'].hist(bins=100)

In [ ]:
afs_sample_enhanced_w_salaries[afs_sample_enhanced_w_salaries['hourly_wage']>20]['hourly_wage'].hist(bins=100)

In [ ]:
# Remove outlying salaries from the data.
# The minimum wage for 16 year olds in 2023 was £5.28 so realistically there shouldn't be hourly pay much lower than this.
afs_sample_enhanced_w_salaries = afs_sample_enhanced_w_salaries[(afs_sample_enhanced_w_salaries['hourly_wage']>=5)&(afs_sample_enhanced_w_salaries['hourly_wage']<50)]

In [ ]:
# Export data for the boxplot of hourly wage by sector
afs_sample_enhanced_w_salaries[['id', 'sector_group', 'hourly_wage']].to_csv('outputs/salary_boxplot.csv')

In [ ]:
# Data for the line plot of median salary by sector over time
afs_sample_enhanced_w_salaries.groupby(['year', 'sector_group']).agg({'hourly_wage': 'median'}).unstack(fill_value=0).to_csv('outputs/hourly_wage_by_year_and_sector.csv')

In [ ]:
afs_sample_enhanced_w_salaries_dimensions = pd.merge(afs_sample_enhanced_w_salaries, dimensions_wide, on='id', how='left')

In [ ]:
len(afs_sample_enhanced_w_salaries_dimensions) - len(afs_sample_enhanced_w_salaries)

In [ ]:
hourly_wage_v_flex_hours = afs_sample_enhanced_w_salaries_dimensions.groupby('sector').agg({'hourly_wage': 'median', 'FLEX_HOURS': 'sum', 'sector': 'size'})
hourly_wage_v_flex_hours['prop_flex_hours'] = hourly_wage_v_flex_hours['FLEX_HOURS'] / hourly_wage_v_flex_hours['sector']
hourly_wage_v_flex_hours

hourly_wage_v_flex_hours.to_csv('outputs/hourly_wage_and_flex_hours_by_sector.csv')

In [ ]:
hourly_wage_v_ld = afs_sample_enhanced_w_salaries_dimensions.groupby('sector').agg({'hourly_wage': 'median', 'L&D': 'sum', 'sector': 'size'})
hourly_wage_v_ld['prop_l&d'] = hourly_wage_v_ld['L&D'] / hourly_wage_v_ld['sector']
hourly_wage_v_ld.to_csv('outputs/hourly_wage_and_l&d_by_sector.csv')

# Contract type

In [ ]:
contract_refs = processed_ads[processed_ads['subcategory']=='CONTRACT']

In [ ]:
contract_refs.columns

In [ ]:
# Check most frequent words and then we'll do a basic regex for these frequently occurring terms
from wordcloud import WordCloud
import matplotlib.pyplot as plt

text = " ".join(
                contract_refs[
                    "sentences_split"
                ].tolist()
                )

# Generate a word cloud image
wordcloud = WordCloud().generate(text)

# Display the generated image:

plt.imshow(wordcloud, interpolation='bilinear')
plt.axis("off")

In [ ]:
analysis_utils.classify_contract_type('Care Assistant  Guaranteed minimum 20-hour contract Salary  £11per hour weekday evenings and  £12 per hour weekends Hours 5 00 pm to 10 00 pm evenings and alternate weekends Location  Chichester, Bognor Regis and surrounding areas.')

In [ ]:
contract_refs['classified_contract_type'] = contract_refs['sentences_split'].apply(analysis_utils.classify_contract_type)
contract_refs['classified_contract_type'].value_counts()

In [ ]:
#Inspect the ones that couldn't be identified
contract_refs[contract_refs['classified_contract_type']=='Unknown']['sentences_split'].to_list()

In [ ]:
contract_refs['classified_contract_type'].value_counts()

Some IDs get matched to more than one contract type, so we need to pick just one for our output.

In [ ]:
counts = contract_refs.groupby(['id', 'classified_contract_type']).size().reset_index(name='count')

# Find the most frequent 'classified_contract_type' for each job
most_frequent = counts.loc[counts.groupby('id')['count'].idxmax()]

# gather all the different contract types that were applied to the same job
contract_df = contract_refs.groupby('id')['classified_contract_type'].agg(list).reset_index()

def determine_contract_type(types: List[str]):
    if 'Temporary' in types:
        return 'Temporary'
    else:
        # Count occurrences of each type
        type_counts = pd.Series(types).value_counts()
        most_common = type_counts.idxmax()
        if len(type_counts) == 1:  # Only one unique type
            return most_common
        elif len(type_counts) > 1:
            # If the list contains more than one type, we need to check the most common
            if most_common == 'Permanent' or most_common == 'Unknown':
                return most_common
            else:
                return 'Unknown'  # Fallback to 'Unknown' if neither 'Temporary' nor most frequent matches
        return 'Unknown'

# Step 5: Apply the function to determine the final contract type for each group
contract_df['final_contract_type'] = contract_df['classified_contract_type'].apply(determine_contract_type)

In [ ]:
contract_df

In [ ]:
afs_sample_enhanced_w_salaries_dimensions_contract = pd.merge(afs_sample_enhanced_w_salaries_dimensions, contract_df[['id', 'final_contract_type']], on='id', how='left')

In [ ]:
afs_sample_enhanced_w_salaries_dimensions_contract['final_contract_type'].value_counts(dropna=False)

In [ ]:
afs_sample_enhanced_w_salaries_dimensions_contract['final_contract_type'] = afs_sample_enhanced_w_salaries_dimensions_contract['final_contract_type'].fillna('Unknown')

In [ ]:
afs_sample_enhanced_w_salaries_dimensions_contract['final_contract_type'].value_counts(dropna=False)

In [ ]:
afs_sample_enhanced_w_salaries_dimensions_contract.groupby(['sector_group', 'final_contract_type']).size()

In [ ]:
contract_prop_df = afs_sample_enhanced_w_salaries_dimensions_contract.groupby(['sector_group', 'final_contract_type']).size().unstack(fill_value=0)
contract_prop_df

In [ ]:
contract_prop_df.to_csv('outputs/sector_contract_type.csv')

# Extracting hours

In [ ]:
hours_refs = processed_ads[processed_ads['subcategory']=='HOURS']
hours_refs

In [ ]:
hours_refs['hours_per_week'] = hours_refs['sentences_split'].apply(analysis_utils.match_hours_per_week)
hours_refs['working_days'] = hours_refs['sentences_split'].apply(analysis_utils.count_working_days)
hours_refs['is_full_time'] = hours_refs['sentences_split'].apply(analysis_utils.check_full_time)
hours_refs

In [ ]:
# Apply the function to each row of the DataFrame
hours_refs['hr_per_week_final'] = hours_refs.apply(analysis_utils.calculate_hr_per_week_final, axis=1)

# Display the updated DataFrame
hours_refs


In [ ]:
hours_refs['hr_per_week_final'].value_counts(dropna=False)